# Lab 4 — Clean the broken sensor feed
### EP0418 · Topic 4 · 2 practical hours

**You need:** nothing to download: the notebook fetches `ep0418.py` and
`smart_building_sensors.csv` itself. Have the printed Lab 4 sheet beside you.

---
**Where this lab fits.** The labs build one system, a stage at a time. Lab 2 brought
data in. Lab 3 put readings on a clock you can trust. This lab makes the values
honest: it is the **clean** stage. In Lab 5 you look for unusual readings in cleaned
data, and in Lab 8 the steps you take today become one function, `clean`, in the
whole system.

![The five stages of the system the labs build: ingest (Lab 2), clean (Labs 3 and 4, Lab 4 highlighted), analyse (Labs 5 and 6), decide and respond (Lab 7)](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/system_lab04.png)

A classroom's sensor logger sent sixty days of readings, and the file is broken in
six ways. Before the readings can be used for anything, each fault has to be found and
fixed without hiding it, and every row that changes has to be explained. The lab does
it in six steps. Each one opens with this picture again, its own step outlined, so you
always know where you are.

![The plan in six steps: first look at the file, Task 1 -999 becomes blank, Task 2 one row per moment, Task 3 a row every 5 minutes, Task 4 explain every row, Task 5 faults no check finds](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan.png)

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **If Gemini says *Daily limit reached*:** paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Prompts** sit in grey quote boxes. When you change one, keep its rules (for
  example, do not drop rows; show the row count before and after): they matter as much
  as the task. A prompt with no example is **zero-shot**; one example of the output
  you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.


> ### This dataset is deliberately broken
> **Six** separate faults are hidden in this file: −999 values that are not
> readings, duplicate timestamps, a five-and-a-half-hour outage, an impossible energy spike, a
> sensor that gets stuck, and a sensor that slowly drifts. The table and the checks
> show you the first four. **The last two you will only find by plotting**, in
> Task 5.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows, {raw.timestamp.min()} -> {raw.timestamp.max()}")

## First look — what is in the file (10 minutes)

![Where you are: step 1 of 6, look at the file](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan_step1.png)

The file comes from one classroom's sensor logger. The cell above printed its size:
17,354 rows, one every 5 minutes from 1 May to 29 June 2026.

**What each field means.** Read this before the first look: you can only spot an
impossible value once you know what each column measures, and in what unit.

| Field | What it means | Example (1 May, 12:15) |
|---|---|---|
| `timestamp` | when the reading was taken, Singapore time, every 5 minutes | 2026-05-01 12:15 |
| `temp_c` | room temperature, in °C | 25.76 |
| `humidity_pct` | how damp the air is, in %: it can only be 0 to 100 | 72.1 |
| `co2_ppm` | carbon dioxide in the air, in parts per million. Outdoor air is about 420; people breathing raise it | 865 |
| `occupancy` | how many people are in the room | 12 |
| `energy_kwh` | electricity the room used in that 5-minute slot, in kWh (the timestamps go up 5 minutes at a time: 00:00, 00:05, 00:10) | 7.668 |

1. **Run the cell below.** It counts blank values and repeated timestamps, then
   prints each column's smallest (`min`), average (`mean`) and largest (`max`) value.

In [ ]:
print("blank values:", raw.isna().sum().sum())
print("duplicated timestamps:", raw.timestamp.duplicated().sum())
print()
print(raw.describe().T[["min", "mean", "max"]].round(1).to_string())

### Read that table before going further

**What you should see:** 0 blank values, 140 duplicated timestamps, and a table with
the smallest, mean and largest value of each column. `isna()` reports 0 blanks, but
the smallest `humidity_pct` is −999.

**You met −999 in Lab 1, in this same file.** Humidity can only be 0 to 100 %, so
−999 is not a reading: it is the logger's code for "no answer", a **sentinel**. The
cell is not empty, so the blank count says 0; and it counts as a number, so the 95 of
them pull the mean humidity down from 75.8 % to the 69.9 % in the table. In Lab 1 you
only saw it. In Task 1 you choose what each one becomes.

1. **In the table the cell above printed**, compare each column's `max`
   with its `mean`.
2. **Find** one other column whose `min` or `max` is physically
   impossible: a value this room could not produce. Hint: a classroom does not use
   eight times its usual energy in five minutes.
3. **Write it** in the box below: the column, min or max, and why it is impossible.
   You do not fix it today: Lab 5 is about spikes.

**Other impossible values (column, min or max, why impossible):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

One good answer: `energy_kwh`, the maximum. Each row covers five minutes: the
timestamps go 00:00, 00:05, 00:10, one row every five minutes. It reaches 33.6 kWh in one five-minute slot, while the mean is 4.3 and the normal daily peak in this room is about 9.2 (the
middle of the 60 daily maximums). That is about eight times the mean and more than
three and a half times the normal peak, and a classroom cannot do that. If you
plot it later you will see it is not one bad row but three short bursts, on 22 May,
3 June and 18 June. A common wrong answer is the `co2_ppm` maximum of about 1,217
ppm. It looks big next to the mean of 609, but a full, poorly ventilated classroom
really can reach that, so it is high, not impossible. A minimum of 0 for `occupancy`
is also fine: the room is empty at night.
</details>

## Task 1 — Choose what each −999 becomes (25 minutes)

![Where you are: step 2 of 6, -999 becomes blank](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan_step2.png)

*Topic 4: zero is not the same as missing. A missing reading must look missing.*

**Why change the −999s?** Every stage after this one reads `humidity_pct`: a chart, a
detector in Lab 5, an alert in Lab 7. Each would take −999 as a real humidity. So each
−999 must become something else before the column is passed on, and what it becomes is
the decision in this task.

**The goal:** `clean`, a copy of `raw` with no −999 left in `humidity_pct`. Gemini
writes the code, but the choices are yours: in this task you also learn what a prompt
must say so that Gemini does not have to guess them.

**Step 1. Answer two questions before you ask the AI.** On 1 May at 12:20 the humidity is −999,
but the same row also holds a real temperature (26.13 °C), CO2 (852 ppm), occupancy
(11) and energy (7.874 kWh). A −999 can be deleted with its whole row, turned into 0,
or turned into a blank (NaN). Write two short answers in the box below:

1. Which should each −999 become: deleted with its whole row, 0, or a blank (NaN)? Why?
2. For the two you did not pick, what goes wrong with each?

**My decisions:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2. Try a one-line prompt, run it, and see what Gemini had to guess.** Open
Gemini (the **spark icon** at the bottom centre of the Colab window) and start a new
chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then
**Yes**). Type this prompt into the chat and press Enter:

> **Prompt:** Remove the -999 values from raw.

When its green cell appears, click **Accept and run**. Then run the cell below. It
finds what Gemini's cell made and prints its name, its rows, and what the −999s
became. It also reloads `raw` from the file, in case Gemini's cell changed it.

In [ ]:
# TODO(you): Accept and run Gemini's green cell for the one-line prompt first, then run this cell.
_changed = len(raw) != 17354 or int((raw["humidity_pct"] == -999).sum()) != 95
_found = False
for _name, _v in list(globals().items()):
    if _name.startswith("_") or not isinstance(_v, pd.DataFrame) or "humidity_pct" not in _v.columns:
        continue
    if _name == "raw" and not _changed:
        continue
    _h = _v["humidity_pct"]
    print(f"Gemini's result: {_name}   rows: {len(_v):,}   -999 left: {int((_h == -999).sum())}   "
          f"blank: {int(_h.isna().sum())}   zero: {int((_h == 0).sum())}")
    _found = True
if not _found:
    print("No result from Gemini found yet: run its green cell first.")
if _changed:
    print("Gemini's cell changed raw itself, so raw was reloaded from the file.")
    raw = pd.read_csv(CSV, parse_dates=["timestamp"])

**Write three short answers** in the box below. Use the line the cell above printed,
the one that starts *Gemini's result:*.

1. **What name did Gemini give its result?** It is the word right after *Gemini's
   result:*, for example `df_clean` or `clean`. Every later cell in this lab looks for
   a table called `clean`: under any other name they say *Not there yet*, even if the
   table is right. Did your prompt ask for that name? (yes or no)
2. **What happened to the −999s?** Write the numbers after *rows*, *-999 left*, *blank*
   and *zero*. The right result has 17,354 rows, 0 left, 95 blank and 0 zero.
3. **Did the answer in the chat say how you could check that the result is right?**
   (yes or no)

**If yours came out right, Gemini guessed right.** Your prompt did not say keep the
rows, use blanks, or call it `clean`. *Remove* can just as well mean delete the 95 rows
(17,259 left) or put 0 in the cells, and the same prompt in a new chat can give either.
You only know yours is right because the cell above counted for you. That is the rule
for any prompt: what you leave out, the AI guesses. The four-part prompt below turns
each guess into an instruction, and says what to count to check it.

**What the one-line prompt left to Gemini:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 3. Write your own prompt, in four parts.** Each part answers one question. The
questions work for any prompt that asks for code. Write one line for each part in the
box *My prompt* below. You do not send it: in Step 4 you compare it with ours.

1. **What I have:** what would someone who has never seen your notebook need to know
   about the data? Ask yourself: to someone who has never seen this logger, does −999
   look like a very dry room, or like no reading? (here: `raw`, and what −999 means in
   `humidity_pct`)
2. **What I want:** what should the result look like, and what is it called? Ask
   yourself: should `raw` itself change, or a copy of it? What should be left where each
   −999 was? (here: a copy called `clean`, with no −999 left)
3. **How I will check:** what printed number would show it worked? Ask yourself: how
   many −999 are there now, and how many blanks should there be after? (here: the rows
   before and after, and how many blanks there are)
4. **My decisions:** your choices from Step 1, written as instructions, shown on one
   real row. Ask yourself: if a stranger did this by hand, what could they do
   differently from you? (here: your two answers from Step 1, shown on the row from
   1 May 12:20: what its humidity becomes, and what happens to the rest of the row)

**My prompt:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 4. Compare with ours, then send ours.** Start a new chat (**⋮** › **New chat** ›
**Yes**), so the one-line prompt does not mix in. Open our prompt below and check it
against yours, part by part.

<details><summary><b>Show the prompt</b> (ours: open it after you have written yours, then copy all of it into the chat and press Enter)</summary>

> **What I have:** a DataFrame `raw`. In its `humidity_pct` column, -999 means "no
> reading": the logger had no answer.
>
> **What I want:** a copy of `raw` called `clean`, with no -999 left in `humidity_pct`.
>
> **How I will check:** show the row count before and after, and how many blanks
> `humidity_pct` has. Explain in plain words what the code does.
>
> **My decisions:** one example: on 1 May at 12:20 the humidity is -999, but the
> temperature (26.13), CO2 (852), occupancy (11) and energy (7.874) are real, so that
> row stays with only its humidity blank. Each -999 in `clean` must become NaN (blank),
> and its row must stay.

**Compare.** Did yours have all four parts? The part people most often leave out is
the last sentence. It says two things:

- **NaN, not 0**: humidity 0 % is a real reading (bone-dry air), so 0 would turn "no
  answer" into a false measurement, and `isna()` would still find no blanks;
- **its row stays**: the other four readings in that row are real. Deleting the 95
  rows would throw away 380 good readings (95 rows × 4 columns).

Send ours, so everyone's check gives the same answer.
</details>

**Why our prompt has a worked example.** It shows Gemini one finished row: 1 May at
12:20 keeps its four real readings, and only its humidity becomes blank. A prompt with
one worked example is called **one-shot**; with no example it is *zero-shot*, and with
two or more, *few-shot*. Gemini copies the example's rule, so the example must be right.

**Step 5.** **Run Gemini's code.**

1. Gemini answers by adding a new code cell to your notebook, with a green edge.
   Click **Accept and run**: the button sits at the bottom of the chat panel, just
   above the box where you type. This keeps the code and runs it.
2. If the code appears only in the chat and no green cell appears: copy the code,
   click **+ Code** at the top of the page, paste it into the new cell and run it.
3. Colab may jump to the green cell, even at the top of the notebook. Scroll back
   down to Task 1 for Step 6.

**Step 6.** **Run the check cell just below.** You should see two `[PASS]`
lines (the expected output is under the cell). `rows_at_least` requires
every one of `raw`'s 17,354 rows: you said what "right" looks like before running
anything. That is called a **known-answer eval**.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `clean`), then run this cell.
clean = keep("clean")
if filled(clean):
    print(f"humidity blanks: {int(clean.humidity_pct.isna().sum())}   zeros: {int((clean.humidity_pct == 0).sum())}")

# ---- verification contract — do not edit ----
check.no_sentinels(clean, cols=["humidity_pct"], sentinel=-999)
check.rows_at_least(clean, len(raw))
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** `humidity blanks: 95   zeros: 0`. If it says 0 blanks and 95 zeros, the −999s became 0 (outcome C below).
- **First check:** `[PASS] no_sentinels(-999)`: no −999 left in `humidity_pct`.
- **Second check:** `[PASS] rows_at_least(17354)`, and under it *17,354 rows.*
- **Last line:** *All checks passed.*

**Step 7.** **Run the cell below.** It draws `humidity_pct` twice: the top
panel is `raw`, the bottom panel is your `clean`. Then find your result in the table
under the chart: outcome A, B or C.

In [ ]:
if filled(clean):
    show.before_after(raw, clean, "humidity_pct", title="Replacing -999 with NaN")

**What you should see:** the top panel has thin spikes down to −999. Compare the
bottom panel and its title with the three outcomes:

| | rows in `clean` | bottom panel | what it means |
|---|---|---|---|
| **A** | 17,354 | small gaps where the spikes were | **Right.** No reading now looks like no reading. |
| **B** | 17,259 | no gaps; the title says *after: 17,259 rows* | **Wrong:** the 95 rows were deleted. `rows_at_least` fails. |
| **C** | 17,354 | spikes down to 0 | **Wrong, and no check catches it:** "no answer" became 0 % humidity. |

**If you got A:** go on to Task 2.

**If you got B or C:** ask Gemini to fix it. In the same chat, type one
sentence with your numbers. For **B**:

> **Prompt:** clean has 17,259 rows but must keep all 17,354. Replace -999 with NaN
> instead of removing the rows. Keep everything else.

For **C**:

> **Prompt:** clean has 17,354 rows, but the -999 values became 0. Humidity 0 is a
> real reading; they must be NaN. Fix that and keep everything else.

Click **Accept and run**, then run the check cell and the chart cell again: the
bottom panel should now have gaps. Give the numbers, not just *it didn't work*:
with the numbers, Gemini knows what to fix; without them, it guesses.

## Task 2 — Remove repeated timestamps and sort by time (15 minutes)

![Where you are: step 3 of 6, one row per moment](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan_step3.png)

**Why remove them?** A moment that appears twice is counted twice: every count,
average and alarm that uses the rows gives it double weight. Some timestamps appear twice because the logger sent the same moment again (a retry). In this file, 140 timestamps appear twice.

**Not the same as Lab 3.** In Lab 3, the 05:00 hour caught two *different* readings,
2,232 taxis at 05:00:00 and 1,766 at 05:59:33: both were real measurements, 59 minutes
apart, so you chose one number to stand for the hour. Here the logger sent *the same
moment twice*, to the second: there was one measurement, and the second row is an
echo of it. So you do not combine them: you keep one copy and drop the other.

**The 140 pairs come in two kinds.** Here are four real rows from early on 24 May, the
same rows as in the clip after the check cell:

| copy | timestamp | temp_c | humidity_pct | co2_ppm | occupancy | energy_kwh |
|---|---|---|---|---|---|---|
| first | 24 May 03:50 | 24.10 | 79.2 | 427 | 0 | 1.493 |
| last | 24 May 03:50 | 24.10 | 79.2 | 427 | 0 | 1.493 |
| first | 24 May 03:55 | 24.78 | blank (was −999) | 423 | 0 | 1.967 |
| last | 24 May 03:55 | 24.78 | 79.4 | 423 | 0 | 1.967 |

- **03:50 is like 138 of the pairs:** the two copies are identical, so keeping either
  one loses nothing.
- **03:55 is like the other 2:** the first copy has no humidity, and the last copy,
  sent again, has the real reading, 79.4. The other pair like it is 9 June at 04:10,
  where the last copy has 77.3.

**Why the last copy, every time.** A logger sends a moment again when it is not sure
the first send worked, so the last copy is its newest attempt. The file agrees: in all
140 pairs the last copy is the same as the first or better, never worse. So "keep the
last copy" is one rule that is right for every pair, and you can check it by counting:
93 humidity blanks after, 2 fewer than the 95 now. The file is already in time order;
sorting after removing the copies makes sure of it.

**The goal:** `clean` with each timestamp once, sorted by time.

**This time you write the prompt and send it.** Use the same four questions as in
Task 1.

**Step 1. Answer two questions.** Write them in the box *My decisions* below:

1. Which copy of a repeated timestamp do you keep, and why? (24 May 03:55: the first
   copy's humidity is blank, the last copy's is 79.4)
2. Should `clean` end up with more rows than now, fewer, or the same? (each of the 140
   repeated timestamps has one copy too many)

**My decisions:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2. Write your prompt.** One line for each of the four parts, in the box
*My prompt* below:

1. **What I have:** the data, as someone who has never seen your notebook needs it.
   Ask yourself: which table is it now, and what has Task 1 already done to it? Which
   column holds the time?
2. **What I want:** the result, and its name. The check cell after Step 4 needs it to
   stay in `clean`. Ask yourself: afterwards, how many times should each timestamp
   appear, and in what order?
3. **How I will check:** what the code should print. Ask yourself: 140 timestamps appear
   twice, so how many rows should be left? Which second number would show that the two
   real humidity readings survived?
4. **My decisions:** your two answers from Step 1, with one real example. Ask yourself:
   which real pair of rows shows which copy to keep?

**My prompt:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 3.** **Send it.** Start a new chat (**⋮** at the top right of the
Gemini panel, then **New chat**, then **Yes**), type your prompt and press Enter. When
the green cell appears, click **Accept and run**: the button sits at the bottom of the
chat panel, just above the box where you type. If the code appears only in the chat,
copy it, click **+ Code** at the top of the page, paste it and run it. If Gemini shows
red and green lines inside the check cell (first line `# TODO(you)`), click
**Cancel**. Then scroll back down to this task.

**Step 4.** **Run the check cell below.** You should see two `[PASS]` lines,
`humidity blanks: 93`, and 17,214 rows (140 fewer).

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it updates `clean`), then run this cell.
clean = keep("clean")

# ---- verification contract — do not edit ----
check.no_duplicate_timestamps(clean, col="timestamp")
check.sorted_in_time(clean, col="timestamp")
check.report()
if filled(clean):
    print(f"rows in clean: {len(clean):,}   humidity blanks: {int(clean.humidity_pct.isna().sum())}")

**What your code in this task just did**, on five real rows from 24 May (it plays
slowly, about a minute):
![Two rows for the same moment: the first copy says -999, the retry 79.4, so keep the last copy](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_duplicates.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **First check:** `[PASS] no_duplicate_timestamps`, and under it *All 17,214 timestamps unique.*
- **Second check:** `[PASS] sorted_in_time`.
- **Last line:** *rows in clean: 17,214   humidity blanks: 93*. That is 140 rows fewer than `raw`'s 17,354, and 2 blanks fewer than in Task 1: the two real readings from the second copies.

In the Decision box at the end, write: kept the last copy · in the 2 pairs that differ, the last copy has the real reading · rows 17,354 → 17,214.

**Step 5.** **Only if a number is wrong.** Find the last line the check cell printed,
*rows in clean: … humidity blanks: …*, in the list below. Copy the sentence under it
into the same chat and press Enter.

- **rows in clean: 17,074.** Both copies of each repeated timestamp
  were dropped, so 140 real moments are gone.

  > **Prompt:** clean has 17,074 rows, but it should have 17,214: both
  > copies of each repeated timestamp were dropped. Keep one copy of each. Fix only that.

- **humidity blanks: 95.** The first copy was kept, so 24 May 03:55 and 9 June 04:10 lost
  their real humidity.

  > **Prompt:** clean has 95 humidity blanks, but it should have 93: the first copy of
  > each repeated timestamp was kept. Keep the last copy instead. Fix only that.

- **Any other numbers.** Type your two numbers into this sentence:

  > **Prompt:** clean has [your rows] rows and [your blanks] humidity blanks, but it
  > should have 17,214 rows and 93 blanks. Fix only that.

Click **Accept and run**, then scroll up and run the check cell again. If it is still
wrong after two tries, open our prompt below and compare it with yours.

<details><summary><b>Our prompt</b> (open it only after you have sent yours)</summary>

> **What I have:** `clean`, a DataFrame with a `timestamp` column. 140 timestamps
> appear twice, because the logger sent the same moment again.
>
> **What I want:** each timestamp once, sorted by time, kept in `clean`.
>
> **How I will check:** show the row count before and after, and how many blanks
> `humidity_pct` has. Explain in plain words what the code does.
>
> **My decisions:** keep the last copy of each repeated timestamp. The logger sends a
> moment again when it is not sure the first send worked, so the last copy is its
> newest attempt. Example: on 24 May at 03:55 the first copy's humidity is blank and the
> last copy's is 79.4, so the row kept has 79.4. Do not drop any other row.
</details>

## Task 3 — Put the readings on a regular 5-minute grid (15 minutes)

![Where you are: step 4 of 6, a row every 5 minutes](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan_step4.png)

**You did this in Lab 3.** There you put the taxi readings on an hourly grid and 39
hours came out empty. Then you looked at the slots next to them: each empty hour had a
neighbour holding two readings. Nothing was lost; the readings had drifted across the
hour. The lesson was: **before you call an empty slot lost data, look at its
neighbours.**

**Same tool, a different file.** Here each slot is 5 minutes, and every reading is
exactly on a 5-minute mark (00:00, 00:05, 00:10 …). Later steps count rows to measure
time (12 rows make an hour), so the grid is needed here too. But first, the Lab 3
question: are the empty slots lost data this time?

**Step 1. Look at the neighbours, as in Lab 3.**

1. **Run the cell below.** It counts how many readings in `clean` fall in each 5-minute
   slot, then prints how many slots hold none, one, or two or more, and the first and
   last empty slot.

In [ ]:
# Count the readings in each 5-minute slot. Nothing is changed.
if filled(clean):
    _t = clean["timestamp"] if "timestamp" in clean.columns else clean.index.to_series()
    _per_slot = pd.Series(1, index=pd.DatetimeIndex(_t)).resample("5min").size()
    print(f"slots: {len(_per_slot):,}   empty: {int((_per_slot == 0).sum())}   "
          f"one reading: {int((_per_slot == 1).sum()):,}   two or more: {int((_per_slot > 1).sum())}")
    _empty = _per_slot[_per_slot == 0].index
    if len(_empty):
        print(f"first empty slot: {_empty.min():%d %b %H:%M}   last empty slot: {_empty.max():%d %b %H:%M}")

**What you should see:** `slots: 17,280   empty: 66   one reading: 17,214   two or
more: 0`, then `first empty slot: 11 Jun 02:00   last empty slot: 11 Jun 07:25`.

**Why the grid needs a rule, such as the mean.** To build the grid, the code must be
told what one number a slot holds when readings fall in it. In Lab 3 that mattered:
the 05:00 hour caught two readings. Here no slot holds two (the printout says *two or
more: 0*), so the mean of a slot is simply its one reading; *first* or *last* would
give the same. The rule matters only for the 66 empty slots. The mean of nothing is
blank, which is what you want. Not every rule does that: *sum* adds up nothing to 0, so
the 11 June outage would become 66 slots of CO2 0 ppm, and no row would look empty.

**Step 2. Answer two questions.** Write them in the box *My decisions* below:

1. In Lab 3 the empty hours were not lost data. Are these 66 empty slots lost data?
   Which two things in the printout tell you? (the number after *two or more*, and the
   first and last empty slot)
2. What should an empty slot hold, such as 11 June at 03:00: 0, the reading before
   copied in, a line drawn between the readings either side, or blank? Pick one and say
   why.

**My decisions:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 3. Write your prompt.** One line for each of the four parts, in the box
*My prompt* below. No answers this time, only four questions you can ask of any
prompt, and the name the check needs:

1. **What I have:** what is the data called, and what is odd about it that a stranger
   would not guess?
2. **What I want:** what comes back, and how many rows should it have? (the check cell
   after Step 4 needs it to be called `grid`)
3. **How I will check:** which number do you already know the right answer should
   print? What would a wrong answer print instead?
4. **My decisions:** your Step 2 answers, written as instructions, shown on one real
   empty slot. Ask yourself: if a stranger did this by hand, what could they put in that
   slot differently from you?

**My prompt:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 4. Send it.** Start a new chat (**⋮** at the top right of the
Gemini panel, then **New chat**, then **Yes**), type your prompt and press Enter. When
the green cell appears, click **Accept and run**. If the code appears only in the chat,
copy it, click **+ Code** at the top of the page, paste it and run it. If Gemini shows
red and green lines inside the check cell (first line `# TODO(you)`), click
**Cancel**. Then scroll back down to this task.

**Step 5. Run the check cell below.** You should see two `[PASS]` lines,
17,280 rows (66 *more* than `clean`), and 66 rows with every value blank.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `grid`), then run this cell.
grid = keep("grid")

# ---- verification contract — do not edit ----
check.regular_cadence(grid, "5min")
check.values_in_range(grid.reset_index() if hasattr(grid, "reset_index") else grid, "temp_c", 15, 40)
check.report()
if filled(grid):
    print(f"rows in grid: {len(grid):,}   rows with every value blank: {int(grid.drop(columns='timestamp', errors='ignore').isna().all(axis=1).sum())}")

**What resampling just did** to the 11 June outage, slot by slot (it plays slowly,
about a minute):
![The 11 June outage on the 5-minute grid: 66 empty slots become rows of NaN, so 17,214 rows become 17,280](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_grid.gif)

**Expected output** *(computed from the data file when this notebook was built)*:

- **First check:** `[PASS] regular_cadence(5min)`, and under it *All 17,279 gaps are exactly 5min.*
- **Second check:** `[PASS] values_in_range(temp_c)`.
- **Last line:** *rows in grid: 17,280   rows with every value blank: 66*. The count goes *up* by 66 from `clean`'s 17,214: the empty slots of the 11 June outage now exist as rows of NaN.

**Step 6.** **Only if a number is wrong.** Find what the check cell printed in the
list below. Copy the sentence under it into the same chat and press Enter.

- **`[FAIL] values_in_range(temp_c)`, and under it *66 value(s) outside the plausible
  range 15 to 40 (actual range 0 to …)*.** The 66 empty slots became 0, so the outage
  now looks like a room at 0 °C.

  > **Prompt:** grid has temperatures of 0 in the 11 June outage, but those slots had no
  > reading: they must stay blank (NaN), not 0. Fix only that.

- **rows with every value blank: 0, and rows in grid: 17,280.** The empty slots were
  filled, with the slot before or with a line between. Both checks can pass anyway: the
  outage is now hidden.

  > **Prompt:** grid has 0 rows with every value blank, but it should have 66: the
  > empty slots of the 11 June outage were filled. A slot with no reading must stay
  > blank (NaN). Fix only that.

- **Any other numbers.** Type your two numbers into this sentence:

  > **Prompt:** grid has [your rows] rows and [your blank rows] rows with every value
  > blank, but it should have 17,280 rows and 66 blank rows. Fix only that.

Click **Accept and run**, then scroll up and run the check cell again. If it is still
wrong after two tries, open our prompt below and compare it with yours.

<details><summary><b>Our prompt</b> (open it only after you have sent yours)</summary>

> **What I have:** `clean`, a DataFrame with a `timestamp` column and the columns
> temp_c, humidity_pct, co2_ppm, energy_kwh and occupancy, one row every 5 minutes,
> except for some hours with no rows.
>
> **What I want:** a regular 5-minute grid with timestamp as the index, taking the
> mean in each slot, called `grid`. No slot holds more than one reading, so the mean is
> that reading, and an empty slot stays blank.
>
> **How I will check:** show the row count before and after, and how many rows have
> every value blank. Explain in plain words what the code does.
>
> **My decisions:** a slot with no reading must stay blank (NaN): not 0, not copied
> from the slot before, not filled in. One example: 11 June at 03:00 had no reading, so
> its row in grid is blank in every column.
</details>

**Run the cell below.** It draws `co2_ppm` from `grid` for the night of the outage
only, 10 June 20:00 to 11 June 12:00, with a dot on every 5-minute slot that holds a
value. The outage hours are shaded.

In [ ]:
if filled(grid):
    try:
        show.series(grid, col="co2_ppm", title="CO2 on the 5-minute grid, the night of 10 to 11 June; shaded: the outage",
                    span=("2026-06-11 02:00", "2026-06-11 07:30"), window=("2026-06-10 20:00", "2026-06-11 12:00"))
    except TypeError:
        print("Your copy of ep0418.py is older than this notebook. Click Runtime, then Disconnect and "
              "delete runtime, then run the cells again from the top.")

**What you should see:** no dots and no line inside the shaded part. The line stops at
01:55 (about 426 ppm) and starts again at 07:30 (about 410 ppm). Those are the 66 empty
slots, and the chart shows them as empty. Outside the shaded part, the night readings
wobble between about 413 and 462 ppm, and from about 08:00 the line climbs as people
arrive.

**What to look out for: dots inside the shaded part.** Any dot there is a value nobody
measured:

- **a row of dots along the bottom, at 0:** the empty slots became 0;
- **a perfectly flat row of dots at about 426:** each empty slot copied the reading
  before it;
- **dots on a dead-straight line from about 426 down to 410:** a line was drawn between
  the readings either side.

Real readings at night wobble; filled-in ones do not. A filled outage at night looks
almost like a quiet night, and that is why filling hides it. If you see any of these,
use Step 6 above. Count them in Task 4: the
row count goes *up* from `clean` to `grid`, and you have to say why.

## Task 4 — Account for every row (10 minutes)

![Where you are: step 5 of 6, explain every row](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan_step5.png)

If you cannot say where every row went, you cannot tell a fix from a loss: a
cleaning step that quietly drops a day looks the same as one that removes copies.
Task 1 kept every row; Tasks 2 and 3 changed the count. Accounting for every row
means finding a cause for each change that you can **count**, and checking that the
count matches exactly. This is called **error analysis**. In Lab 8 you do the same for
the whole system: every stage, rows in and rows out.

**Step 1. Run the cell below.** It prints the row count of `raw`, `clean` and `grid`,
and how much it changed in each task. Nothing is changed.

In [ ]:
if filled(clean, grid):
    print(f"Task 2: raw {len(raw):,} -> clean {len(clean):,}   ({len(clean) - len(raw):+,} rows)")
    print(f"Task 3: clean {len(clean):,} -> grid {len(grid):,}   ({len(grid) - len(clean):+,} rows)")

**What you should see:** `Task 2: raw 17,354 -> clean 17,214   (-140 rows)` and
`Task 3: clean 17,214 -> grid 17,280   (+66 rows)`.

**Step 2. Answer two questions.** Write them in the box *Where the rows went* below.
Look back at Task 2's opening and at the printout in Task 3 Step 1.

1. **The 140 rows Task 2 removed.** Which rows were they? How do you know that no real
   reading was lost with them?
2. **The 66 rows Task 3 added.** Which rows are they, and what is in them? Is any of it
   a reading?

**Where the rows went:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 3. Run the check cell below.** You do not type anything in it. It counts the
two causes itself, the repeated timestamps in `raw` and the slots in `grid` where every
value is blank, and checks that each cause explains its change in rows exactly. A row
that no cause explains was lost, or made up, for a reason nobody chose.

In [ ]:
# ---- verification contract — do not edit ----
if filled(clean, grid):
    _values = grid.drop(columns="timestamp", errors="ignore")
    check.rows_reconciled(raw, clean, cause=-int(raw.timestamp.duplicated().sum()),
                          why="repeated timestamps in raw", names=("raw", "clean"))
    check.rows_reconciled(clean, grid, cause=int(_values.isna().all(axis=1).sum()),
                          why="empty slots in grid (every value blank)", names=("clean", "grid"))
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **First check:** `[PASS] rows_reconciled(raw -> clean)`, and under it *raw 17,354 -> clean 17,214 (-140): all 140 explained by 140 repeated timestamps in raw.*
- **Second check:** `[PASS] rows_reconciled(clean -> grid)`, and under it *clean 17,214 -> grid 17,280 (+66): all 66 explained by 66 empty slots in grid (every value blank).*
- **If the first says *140 row(s) unaccounted for*:** Task 2 removed both copies of each repeated timestamp, so 140 real moments are gone. Go back to Task 2 Step 5.
- **If the first says *93 row(s) unaccounted for*:** Task 1 deleted the 95 rows with −999 instead of blanking them (2 of those rows were repeated copies anyway, so 93 real readings are gone). Go back to Task 1.
- **If the second says *66 row(s) unaccounted for*:** Task 3 filled the empty slots, so 66 rows hold values nobody measured. Go back to Task 3 Step 6.

## Task 5 — Find the two faults no check can catch (20 minutes)

![Where you are: step 6 of 6, faults no check finds](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_plan_step6.png)

Two faults in this file pass every check, because every value is in range, none is
missing and none is repeated: a sensor that **stops changing** (it sends exactly the
same value, slot after slot), and a sensor that **drifts** (its readings creep up a
little more each day although the room has not changed, so a fixed alarm level would
go off more and more often).

Today you find them by eye. In Lab 5 you build detectors, and you will see which of
them these two faults fool.

1. **Run the cell below.** It draws four sensor channels over the sixty days.
2. **Read the paragraph under the charts** (it starts *What to look for*).
   Then find each of the two faults it describes in the four charts.
3. **Confirm the flat stretch with Gemini**, even if you think you saw it: sixty days
   on one axis hides details. You want, for each of the four channels, the longest run
   of exactly the same value, one row after another: how many rows long it is, and the
   date it starts. **There is no prompt to copy this time.** Write your own in the box
   *My prompt* under the charts (the four questions are just above that box), then start
   a new chat (**⋮** at the top right of the Gemini panel, then **New chat**,
   then **Yes**), type your prompt and press Enter. Gemini adds a new code cell with a
   green edge: click **Accept and run** at the bottom of the chat panel. The answer is
   printed under that green cell (Colab scrolls you there; scroll back to Task 5
   afterwards). The stuck sensor is the one whose run is about 168 rows (14 hours of
   5-minute readings); short runs of a few rows are ordinary quiet stretches. If your
   answer has no run near 168 rows, open our prompt under your box and compare.
4. **Write two answers** in the two boxes at the end of this task: which channel was
   stuck, on which date, and for how many rows (Gemini's run of about 168 rows); and which
   channel drifted, from about which date (the chart whose line climbs in the last weeks
   and does not come back down).

In [ ]:
# Draw all four channels. The finding is yours; the drawing is not the exercise.
if filled(grid):
    for col in ["temp_c", "humidity_pct", "co2_ppm", "energy_kwh"]:
        show.series(grid, col=col, title=col)

**What to look for.** You should see four charts, one per channel. In one channel, a stretch that is perfectly flat; in another, a slow
climb over the last two and a half weeks that never comes back down. Neither is out of
range, neither is a duplicate, neither is missing. Every check in the library
passes on both.

**Your prompt for step 3, in four parts.** One line for each, with the same four
questions as Task 3:

1. **What I have:** what is the data called, and which columns does it have?
2. **What I want:** what comes back? (one line per channel)
3. **How I will check:** what would a stuck sensor print, and what would an ordinary
   quiet stretch of a few rows print?
4. **My decisions:** should `grid` change? Do two blanks in a row count as the same
   value?

**My prompt:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Our prompt</b> (open it only after you have sent yours)</summary>

> **What I have:** `grid`, a DataFrame on a 5-minute grid with the columns temp_c,
> humidity_pct, co2_ppm and energy_kwh.
>
> **What I want:** for each of the four columns, the longest run of identical
> consecutive values: its length in rows, and the date it starts.
>
> **How I will check:** print one line per column. A stuck sensor shows as a run of
> more than an hour (12 rows); a quiet stretch is a few rows. Explain in plain words
> what the code does.
>
> **My decisions:** do not change grid, and blanks do not count as identical values.
</details>

**Stuck sensor (channel, dates, what it was doing):**

**Drifting sensor (channel, dates, what it was doing):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Stuck: `temp_c` on 14 May. It reads exactly 23.87 °C from 09:00 to 22:55, which is
168 slots in a row, about 14 hours. A real room warms up through a school day and
cools at night; this line does not move at all. Drifting: `co2_ppm` from about 12
June to the end of the file on 29 June. At night, when the room is empty, it reads
about 420 ppm (outdoor air) until 10 June, then creeps up to about 570 ppm by 28 June
and never comes back. Outdoor air has
not changed, so the sensor has. A common wrong answer names the `energy_kwh` spikes
or the 11 June outage. Those are faults, but the checks and the table already showed
them. This box is for the two you could only see by plotting. In Lab 5 you will build a
detector for the stuck sensor, and see a fixed CO2 alarm fooled by the drift.

**Both faults on a chart**, slowly (about a minute):
![The stuck temperature sensor on 14 May and the drifting CO2 sensor in June: normal values, so no check fires](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_silent_faults.gif)
</details>

---
## Predict: How many dry days next week? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** On how many of the seven days of the target week the island-average rain is zero (under 0.05 mm). The target is the week of Monday 2026-11-16.

**How it works:** you enter a short function, `predict(history)`, not a number. The league page runs it on the newest data, and the number it gives when you click
**Submit** is your entry: it does not change after you submit. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=4

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*the share of dry days in the last sixty days, times seven, rounded*, on the lab's own file. The league page uses newer data,
so its number will differ.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

def predict(history):
    # the baseline: the share of dry days in the last sixty days, times seven
    dry = history["rain_mm_mean"].tail(60) < 0.05
    print("dry days in the last sixty:", int(dry.sum()))
    return float(round(dry.mean() * 7))

print("The baseline predicts", round(predict(rain), 1), "dry days")

**What you should see:** *The baseline predicts 1.0 dry days*.

**Step 2.** **Write the prompt in four parts, then send it.** The same four parts as Lab 3,
because they answer the four things Gemini cannot know for you:

| Part | The question it answers | Here |
|---|---|---|
| What I have | What is the data, and what does it not hold? | `history`, its columns, and the dates it covers |
| What I want | What should come back, and what is it called? | a function `predict(history)` that returns one number |
| How I will check | What printed line would show it worked? | the data span printed; one sentence on what was dropped |
| My decisions | Where could two people do this differently? | **your idea**, in plain words |

**Why four parts?** Gemini cannot know what data you have, what shape you want back,
how you will check it, or what you decided. Parts 1 to 3 are facts about the contest, so they
are written for you below. Part 4 is yours: replace *MY IDEA* with your idea in one or
two plain sentences (for example "use only the last four Mondays", or "last Monday's reading,
nudged by this week's trend"). If you have no idea yet, write "the simple rule" and improve
it after Step 4.

Start a new chat (**⋮** › **New chat** › **Yes**), paste the prompt with your part 4 filled in, and press Enter.

> **What I have:** a pandas table `history` with the columns date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting, oldest first. It holds one row per day from 26 May 2026 up to the day before entries close (about 160 to 240 rows): nothing earlier, no other columns, no forecast, no internet.
> **What I want:** a function predict(history) that returns one number with float(...): on how many of the seven days of the target week the island-average rain is zero (under 0.05 mm). Only the function and the import lines it needs, in a new cell; do not change the cells above.
> **How I will check:** the first lines of the function print the first date, the last date and the row count. No internet calls and no try/except that hides an error. After the code, one sentence on the idea and one naming anything you dropped from my idea and why.
> **My decisions:** MY IDEA. Check it against what history holds before you write: if it needs something that is not there, say so, then shrink the idea or fall back to the simple rule (the share of dry days in the last sixty days, times seven, rounded). Never write a branch that can only run on data that is not there.

**My idea (part 4), in plain words:**

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. If Gemini shows red and green lines inside the Step 1 cell, click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(rain, "dry days")

**What you should see:** *Your predict says* and a number, usually between 0 and 7, then your
code between two dashed lines. If it says *Gemini's predict has not run yet*, do what that message says.
If you see a red error, paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Gemini?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so it needs a longer prompt that spells out every
rule the league checks. Use it in place of Steps 2 and 3:

1. **Copy the prompt** in the grey box below.
2. **Paste it** into the other AI's chat box. Do not send it yet.
3. **Write your idea.** In the line *My idea: WRITE YOUR IDEA HERE*, replace *WRITE YOUR IDEA HERE* with your idea, in plain words.
   Plain text is fine: no bold or italics needed. If you have no idea yet, delete the whole line: the AI then chooses one.
4. **Optional:** to let the AI look at the numbers first, download the data,
   https://stanleysp-eng.github.io/EP0418-labs/data/rain_daily.csv (the same file your
   function receives, updated every night), and attach it to the chat.
5. **Send it.** Copy the code the AI gives back, then do **Step 4** below: paste that
   code into the league page's **Code** box, in place of the code from Step 3.

```text
I am entering a prediction contest. Write one Python function for it.

The question: On how many of the seven days of the target week the island-average rain is zero (under 0.05 mm). The target is the week of Monday 2026-11-16 to Sunday 2026-11-22.
The simple rule to beat: the share of dry days in the last sixty days, times seven, rounded.
My idea: WRITE YOUR IDEA HERE

First, check my idea against the data before you write anything. history holds one row per day from 26 May 2026 up to the day before I submit (about 160 to 240 rows). There is nothing earlier, no other columns, and no weather forecast or internet. If my idea needs something history does not have (past years, a season from another year, wind, temperature, a live feed), say so plainly, then either shrink the idea to what the data can support or fall back to the simple rule. Never write a branch that can only run on data that is not there.

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: date, rain_mm_mean, rain_mm_max, stations_reporting. What they hold: date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting. The date column is a pandas datetime, one row per day, with no time zone.
3. Do not read any file, do not call the internet and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. The first lines of the function print the first date, the last date and the number of rows, so I can see what the code was given.
5. No try/except that hides a problem. If something fails, let it fail so I can see it.
6. Return one number with return float(...). Not a list, a table, text, None or NaN.
7. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
8. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code: one sentence on what the code really does with this data, and one sentence naming any part of my idea you dropped and why. If you dropped nothing, say so.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=4 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: the page's number is the one that counts. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* and *Your entry is ...* confirm it. Submit again any time before the close to replace it; the latest one counts.

**Write it down** (on the sheet, or double-click here): your entry (the number the league page gave at Submit),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (1.0 dry days), and an idea that uses the data. 11 of the 60 days in the file were dry, so the baseline says 1 (1.28, rounded). Dry days come in runs, so a week can have none or several. Ideas that use this: count the dry days in each recent week and average those, or weight the last two weeks more.
The common weak answer is "Gemini wrote it". It says who wrote the code, not what the idea is. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

An assistant was asked to put the cleaned sensor data onto a 5-minute grid with no blank values left. It wrote the code below.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it, and answer Question 1** in the box under the code cell, or on the
   sheet. Do not run it yet.
2. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and compare what it prints with your answer.
3. **Answer Question 2.**

**Question 1.** The school's facilities team will use `their_grid` to see when the classroom air was stale, from `co2_ppm`. What does `interpolate()` put into the 66 empty slots of the 11 June outage, from 02:00 to 07:25? What would the team believe about the air in those hours, and what do we actually know about them?

**Question 2.** What one check would have caught this before anyone used the result? Say what you would print or look at, and what it would show.

<details><summary>Hint 1: where to look</summary>

What does `interpolate()` do to a run of empty slots between two real readings?

</details>
<details><summary>Hint 2: closer to the answer</summary>

It draws a straight line through them. The 66 empty slots of the 11 June outage get invented values, and so does every blank humidity value. The row count does not change, so `rows_accounted_for` cannot see it, and the Task 3 checks still pass.

![interpolate() draws a straight line through the 11 June outage: 66 invented readings, and every check still passes](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab04_interpolate.gif)

</details>



**Optional:** paste the code into a new Gemini chat with the prompt *Explain this code
in plain words.* Does its explanation name the fault? Often it does not: an
explanation that reads well is not evidence the code is right. To fix the code, type
*Fix the problem in this code that Hint 2 describes. Say in one sentence what you
changed.* If that one change matches Hint 2, click **Accept and run**; if it changed
other things too, click **Cancel** and type *fix only that, nothing else*. You should
see: the second blank count is no longer 0: the outage and the humidity blanks stay blank.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_grid = clean.set_index("timestamp").resample("5min").mean()
# print("blank values before interpolate:", int(their_grid.isna().sum().sum()))
# their_grid = their_grid.interpolate()
# print("blank values after interpolate :", int(their_grid.isna().sum().sum()))
# check.reset(); check.regular_cadence(their_grid, "5min"); check.values_in_range(their_grid.reset_index(), "temp_c", 15, 40); check.report()

**Question 1, my answer:**

**Question 2, my answer:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: the ways a missing value was disguised in today's file (-999, blank, missing rows); why duplicate timestamps appear and which copy to keep; why every row you remove must be accounted for; what rows_accounted_for checks and what it cannot see; what a stuck sensor looks like and why no check catches it. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Questions to close the lab

**Write two answers** in the box below, or on the last page of the sheet. Each one
asks you to use what you saw today, not to repeat a definition.

1. In Lab 3 the 39 empty hours were not lost data; in this lab the 66 empty slots on
   11 June are. Which number in the Task 3 printout told you the difference? If those
   66 slots had been filled with 0 instead, what would someone reading the CO2 chart
   for that night wrongly conclude?
2. The stuck temperature on 14 May passed every check in this lab. Describe, in words,
   one check that would have caught it, and what it would print for 14 May.

**My two answers:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

1. The count after *two or more*. In Lab 3, 39 slots held two readings, one beside
   each empty hour: the readings had moved, not gone. Here it is 0, and the 66 empty
   slots are one block, 02:00 to 07:25 on 11 June: for five and a half hours the
   logger sent nothing. Filled with 0, the chart would drop from about 426 ppm to 0
   and back to about 410: it would look as if the room had no air, or as if the
   sensor had read something, when it read nothing. The day's average CO2 would fall
   from 737 to 568 ppm. Blank says "no reading"; 0 says "a reading of zero".
2. A check for a long run of exactly the same value, for example "no channel may
   repeat one value for more than 12 rows (one hour)". For 14 May it would print
   `temp_c`: 168 rows of 23.87, starting at 09:00. Lab 5 builds this as a detector.
   A range check cannot catch it: 23.87 °C is a normal room temperature.
</details>

---
## Before you leave

**What you hand on.** Your `grid` is the clean stage of the system: 17,280 slots, one
every 5 minutes, with three decisions built in: each −999 became blank, each repeated
moment kept its last copy, and the 66 slots of the 11 June outage stay blank. Lab 5
starts from the first two of these decisions, already done for you, and in Lab 8 these
steps become the `clean` function.

Double-click the box cell below and type there (or write on the last page of the Lab 4
sheet): one **mistake caught** line per AI mistake you caught today, and one
**decision** line for each fault you handled (the −999s, the duplicates, the outage
gap), saying what you did and why. The Task 2 expected output gives one example.

**Mistake caught:** *(three short lines: 1. what I expected, 2. what was there instead, 3. how I found it. If Gemini made no mistake today, use the read-before-you-run code above instead.)*

**Decision:** *(what I decided · why · the evidence)*

*(Your answer box: double-click this cell. Type your answer here.)*